# IWM Historical Stock Price Analysis
## Technical Indicators and Trading Signal Generation

This notebook provides an interactive analysis of IWM stock data with various technical indicators and put/call signal generation based on price movement patterns.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, time
import os
import glob
from typing import Tuple, List, Dict
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)

In [ ]:
# Import the analyzer class from our script
from iwm_analysis import IWMAnalyzer

# Initialize analyzer
analyzer = IWMAnalyzer()

## Step 1: Combine CSV Files

In [ ]:
# Define paths
input_folder = "/workspace/data/stock_prices"
output_file = "/workspace/data/historical_iwm_0824_0825.csv"

# Combine all CSV files
df = analyzer.combine_csv_files(input_folder, output_file)

# Display basic info
print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Check data quality
print("Missing values per column:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

print("\nBasic statistics:")
df.describe()

## Step 2: Calculate Technical Indicators

In [ ]:
# Add all technical indicators
df = analyzer.add_technical_indicators(df)

# Display sample of data with indicators
print("Sample data with indicators:")
df[['Time', 'Last', 'Volume', 'ATR14_W', 'RSI14_W', 'EMA9', 'EMA20', 'EMA50', 'VWAP']].tail(20)

## Step 3: Visualize Price and Indicators

In [ ]:
# Create subplots for visualization
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

# Plot 1: Price with EMAs
axes[0].plot(df['Time'], df['Last'], label='Price', color='black', linewidth=1)
axes[0].plot(df['Time'], df['EMA9'], label='EMA9', color='blue', alpha=0.7)
axes[0].plot(df['Time'], df['EMA20'], label='EMA20', color='orange', alpha=0.7)
axes[0].plot(df['Time'], df['EMA50'], label='EMA50', color='red', alpha=0.7)
axes[0].plot(df['Time'], df['VWAP'], label='VWAP', color='purple', alpha=0.7, linestyle='--')
axes[0].set_ylabel('Price ($)')
axes[0].legend(loc='best')
axes[0].set_title('IWM Price with Moving Averages and VWAP')
axes[0].grid(True, alpha=0.3)

# Plot 2: Volume and RVOL
axes[1].bar(df['Time'], df['Volume'], alpha=0.3, color='gray', label='Volume')
ax1_twin = axes[1].twinx()
ax1_twin.plot(df['Time'], df['RVOL20'], label='RVOL20', color='green', linewidth=2)
ax1_twin.axhline(y=1, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Volume')
ax1_twin.set_ylabel('RVOL')
axes[1].legend(loc='upper left')
ax1_twin.legend(loc='upper right')
axes[1].set_title('Volume and Relative Volume')
axes[1].grid(True, alpha=0.3)

# Plot 3: RSI
axes[2].plot(df['Time'], df['RSI14_W'], label='RSI(14)', color='blue', linewidth=2)
axes[2].axhline(y=70, color='red', linestyle='--', alpha=0.5, label='Overbought')
axes[2].axhline(y=30, color='green', linestyle='--', alpha=0.5, label='Oversold')
axes[2].fill_between(df['Time'], 30, 70, alpha=0.1, color='gray')
axes[2].set_ylabel('RSI')
axes[2].set_ylim(0, 100)
axes[2].legend(loc='best')
axes[2].set_title('Relative Strength Index (Wilder)')
axes[2].grid(True, alpha=0.3)

# Plot 4: Stochastic RSI
axes[3].plot(df['Time'], df['StochRSI_K'], label='StochRSI %K', color='blue', linewidth=2)
axes[3].plot(df['Time'], df['StochRSI_D'], label='StochRSI %D', color='red', linewidth=2)
axes[3].axhline(y=80, color='red', linestyle='--', alpha=0.5)
axes[3].axhline(y=20, color='green', linestyle='--', alpha=0.5)
axes[3].fill_between(df['Time'], 20, 80, alpha=0.1, color='gray')
axes[3].set_ylabel('StochRSI')
axes[3].set_ylim(0, 100)
axes[3].legend(loc='best')
axes[3].set_title('Stochastic RSI')
axes[3].set_xlabel('Time')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 4: Generate Trading Signals

In [ ]:
# Identify price runs
runs = analyzer.identify_runs(df)
print(f"Total runs identified: {len(runs)}")

# Analyze run statistics
up_runs = [r for r in runs if r['direction'] == 'up']
down_runs = [r for r in runs if r['direction'] == 'down']

print(f"\nUp runs: {len(up_runs)}")
print(f"Down runs: {len(down_runs)}")

# Duration statistics
up_durations = [r['duration_minutes'] for r in up_runs]
down_durations = [r['duration_minutes'] for r in down_runs]

print(f"\nUp run durations - Mean: {np.mean(up_durations):.2f}, Median: {np.median(up_durations):.2f}")
print(f"Down run durations - Mean: {np.mean(down_durations):.2f}, Median: {np.median(down_durations):.2f}")

In [ ]:
# Generate trading signals
signals_file = "/workspace/data/historical_iwm_0824_0825_signals.csv"
signals_df = analyzer.generate_signals(df, runs)

# Save signals
signals_df.to_csv(signals_file, index=False)
print(f"Signals saved to {signals_file}")

# Display signal summary
print(f"\nTotal signals generated: {len(signals_df)}")
print(f"Call signals: {len(signals_df[signals_df['trade_type'] == 'call'])}")
print(f"Put signals: {len(signals_df[signals_df['trade_type'] == 'put'])}")

# Display first few signals
print("\nFirst 5 signals:")
signals_df[['trade_type', 'entry_timestamp', 'exit_timestamp', 'entry_price', 'exit_price', 'return_pct']].head()

## Step 5: Analyze Signal Performance

In [ ]:
# Performance analysis
print("Signal Performance Analysis:")
print("="*50)

# Overall statistics
print(f"\nOverall:")
print(f"Average return: {signals_df['return_pct'].mean():.3f}%")
print(f"Median return: {signals_df['return_pct'].median():.3f}%")
print(f"Std dev: {signals_df['return_pct'].std():.3f}%")
print(f"Win rate: {(signals_df['return_pct'] > 0).mean()*100:.1f}%")

# By trade type
for trade_type in ['call', 'put']:
    type_df = signals_df[signals_df['trade_type'] == trade_type]
    print(f"\n{trade_type.capitalize()} signals:")
    print(f"Count: {len(type_df)}")
    print(f"Average return: {type_df['return_pct'].mean():.3f}%")
    print(f"Win rate: {(type_df['return_pct'] > 0).mean()*100:.1f}%")

In [ ]:
# Visualize signal returns distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution of returns
axes[0, 0].hist(signals_df['return_pct'], bins=30, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Return (%)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Signal Returns')

# Returns by trade type
signals_df.boxplot(column='return_pct', by='trade_type', ax=axes[0, 1])
axes[0, 1].set_xlabel('Trade Type')
axes[0, 1].set_ylabel('Return (%)')
axes[0, 1].set_title('Returns by Trade Type')

# Duration vs Return scatter
for trade_type, color in [('call', 'green'), ('put', 'red')]:
    mask = signals_df['trade_type'] == trade_type
    axes[1, 0].scatter(signals_df.loc[mask, 'duration_minutes'], 
                       signals_df.loc[mask, 'return_pct'],
                       color=color, alpha=0.6, label=trade_type)
axes[1, 0].set_xlabel('Duration (minutes)')
axes[1, 0].set_ylabel('Return (%)')
axes[1, 0].set_title('Duration vs Return')
axes[1, 0].legend()
axes[1, 0].axhline(y=0, color='black', linestyle='--', alpha=0.3)

# Cumulative returns
signals_df_sorted = signals_df.sort_values('entry_timestamp')
signals_df_sorted['cumulative_return'] = (1 + signals_df_sorted['return_pct']/100).cumprod()
axes[1, 1].plot(range(len(signals_df_sorted)), signals_df_sorted['cumulative_return'])
axes[1, 1].set_xlabel('Signal Number')
axes[1, 1].set_ylabel('Cumulative Return')
axes[1, 1].set_title('Cumulative Returns')
axes[1, 1].axhline(y=1, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Step 6: Analyze Indicator Values at Entry/Exit

In [ ]:
# Analyze RSI levels at entry for different trade types
print("RSI Analysis at Entry:")
print("="*40)

for trade_type in ['call', 'put']:
    type_df = signals_df[signals_df['trade_type'] == trade_type]
    print(f"\n{trade_type.capitalize()} entries:")
    print(f"Average RSI: {type_df['entry_RSI14_W'].mean():.2f}")
    print(f"Median RSI: {type_df['entry_RSI14_W'].median():.2f}")
    print(f"RSI < 30: {(type_df['entry_RSI14_W'] < 30).sum()} signals")
    print(f"RSI > 70: {(type_df['entry_RSI14_W'] > 70).sum()} signals")

In [ ]:
# Create indicator comparison at entry
indicators_to_compare = ['entry_RSI14_W', 'entry_RVOL20', 'entry_ATR14_W']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, indicator in enumerate(indicators_to_compare):
    for trade_type, color in [('call', 'green'), ('put', 'red')]:
        type_df = signals_df[signals_df['trade_type'] == trade_type]
        axes[idx].hist(type_df[indicator].dropna(), bins=20, alpha=0.5, 
                      label=trade_type, color=color, edgecolor='black')
    
    axes[idx].set_xlabel(indicator.replace('entry_', ''))
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f'{indicator.replace("entry_", "")} at Entry')
    axes[idx].legend()

plt.tight_layout()
plt.show()

## Step 7: Save Enhanced Data

In [ ]:
# Save the complete dataset with indicators
enhanced_file = output_file.replace('.csv', '_with_indicators.csv')
df.to_csv(enhanced_file, index=False)
print(f"Enhanced data saved to: {enhanced_file}")

# Create a summary report
summary = {
    'Data Range': f"{df['Time'].min()} to {df['Time'].max()}",
    'Total Records': len(df),
    'Total Signals': len(signals_df),
    'Call Signals': len(signals_df[signals_df['trade_type'] == 'call']),
    'Put Signals': len(signals_df[signals_df['trade_type'] == 'put']),
    'Average Signal Return': f"{signals_df['return_pct'].mean():.3f}%",
    'Win Rate': f"{(signals_df['return_pct'] > 0).mean()*100:.1f}%",
    'Files Created': [
        output_file,
        enhanced_file,
        signals_file
    ]
}

print("\n" + "="*50)
print("ANALYSIS SUMMARY")
print("="*50)
for key, value in summary.items():
    if isinstance(value, list):
        print(f"{key}:")
        for item in value:
            print(f"  - {item}")
    else:
        print(f"{key}: {value}")